# Test Azure OpenAI API

Notebook đọc cấu hình từ file `.env` ở thư mục gốc của repo.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / ".env").exists():
            return path
    raise FileNotFoundError("Không tìm thấy file .env của repo")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=False)

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "").strip()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip().rstrip("/")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "").strip()
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "").strip()

config = {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
}
missing = [name for name, value in config.items() if not value]

print(f"Đã đọc cấu hình từ: {ENV_PATH}")
for name, value in config.items():
    status = "đã cấu hình" if value else "chưa cấu hình"
    print(f"{name}: {status}")

if missing:
    raise RuntimeError(f"Thiếu biến môi trường: {', '.join(missing)}")

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
)


## Liệt kê model resource hỗ trợ

Danh sách dưới đây là các model resource có thể nhìn thấy. Model vẫn cần được **deploy** trước khi gọi inference.

In [ ]:
available_models = sorted(model.id for model in client.models.list())
print(f"Azure resource trả về {len(available_models)} model entries:\n")
for model_id in available_models:
    print(model_id)


## Test deployment

Azure yêu cầu `model=` là **tên deployment đã tạo**, không phải model ID trong danh sách trên. Xem tên chính xác tại **Microsoft Foundry → Models + endpoints → Deployments**, rồi gán tên đó cho `AZURE_OPENAI_DEPLOYMENT` trong `.env`.

In [ ]:
response = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT,
    messages=[
        {"role": "system", "content": "Bạn là trợ lý trả lời ngắn gọn bằng tiếng Việt."},
        {"role": "user", "content": "Xin chào. Hãy xác nhận Azure OpenAI API đang hoạt động."},
    ],
    max_tokens=200,
)

print(response.choices[0].message.content)
print("Token usage:", response.usage)
